## 1. Load Resume Dataset

In [1]:
import pandas as pd

df = pd.read_csv("../data/resumes/Resume.csv")

In [2]:
df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [3]:
df.columns

Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='str')

In [4]:
df.shape

(2484, 4)

## 2. Explore Resume Dataset

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   ID           2484 non-null   int64
 1   Resume_str   2484 non-null   str  
 2   Resume_html  2484 non-null   str  
 3   Category     2484 non-null   str  
dtypes: int64(1), str(3)
memory usage: 52.3 MB


In [6]:
df.isnull().sum()

ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64

In [7]:
df["Category"].value_counts()

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [8]:
print(df["Resume_str"].iloc[0][:2000])

         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service.         Highlights         Focused on customer satisfaction  Team management  Marketing savvy  Conflict resolution techniques     Training and development  Skilled multi-tasker  Client relations specialist           Accomplishments      Missouri DOT Supervisor Training Certification  Certified by IHG in Customer Loyalty and Marketing by Segment   Hilton Worldwide General Manager Training Certification  Accomplished Trainer for cross server hospitality systems such as    Hilton OnQ  ,   Micros    Opera PMS   , Fidelio    OPERA    Reservation System (ORS) ,   Holidex    Completed courses and seminars in customer service, sales strategies, inventory control, loss preve

## 3. Prepare Resume Text

In [9]:
rag_df = df[["ID", "Resume_str", "Category"]].copy()

In [10]:
rag_df = rag_df.rename(columns={
    "Resume_str": "resume_text"
})

In [11]:
rag_df.head()

,ID,resume_text,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...",HR
2,33176873,HR DIRECTOR Summary Over 2...,HR
3,27018550,HR SPECIALIST Summary Dedica...,HR
4,17812897,HR MANAGER Skill Highlights ...,HR


In [12]:
rag_df = rag_df.dropna(subset=["resume_text"])

rag_df = rag_df[
    rag_df["resume_text"].str.strip() != ""
].reset_index(drop=True)

In [13]:
print("Prepared resumes:", rag_df.shape)

Prepared resumes: (2483, 3)


In [14]:
import re

def clean_resume_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [15]:
rag_df["clean_resume"] = rag_df["resume_text"].apply(clean_resume_text)

In [16]:
print(rag_df["clean_resume"].iloc[0][:2000])

HR ADMINISTRATOR/MARKETING ASSOCIATE HR ADMINISTRATOR Summary Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management. Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commitment to customer service. Highlights Focused on customer satisfaction Team management Marketing savvy Conflict resolution techniques Training and development Skilled multi-tasker Client relations specialist Accomplishments Missouri DOT Supervisor Training Certification Certified by IHG in Customer Loyalty and Marketing by Segment Hilton Worldwide General Manager Training Certification Accomplished Trainer for cross server hospitality systems such as Hilton OnQ , Micros Opera PMS , Fidelio OPERA Reservation System (ORS) , Holidex Completed courses and seminars in customer service, sales strategies, inventory control, loss prevention, safety, time management, leadership and performance assessment. Experience HR Adm

In [17]:
rag_df[["ID", "Category", "clean_resume"]].head()

,ID,Category,clean_resume
0,16852973,HR,HR ADMINISTRATOR/MARKETING ASSOCIATE HR ADMINI...
1,22323967,HR,"HR SPECIALIST, US HR OPERATIONS Summary Versat..."
2,33176873,HR,HR DIRECTOR Summary Over 20 years experience i...
3,27018550,HR,"HR SPECIALIST Summary Dedicated, Driven, and D..."
4,17812897,HR,HR MANAGER Skill Highlights HR SKILLS HR Depar...


## 4. Generate Resume Embeddings

In [18]:
from sentence_transformers import SentenceTransformer

e:\NLP\rag-talent-search-engine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7478.03it/s]


In [20]:
resume_embeddings = embedding_model.encode(
    rag_df["clean_resume"].tolist(),
    show_progress_bar=True
)

Batches: 100%|██████████| 78/78 [01:10<00:00,  1.11it/s]


In [21]:
print(resume_embeddings.shape)

(2483, 384)


In [22]:
print(resume_embeddings[0][:10])

[-0.04447829  0.02122919  0.00516218  0.08680715 -0.01907622  0.00078137
  0.02372742 -0.04067285 -0.10769627 -0.04174348]


## 5. Build FAISS Vector Database

In [23]:
import faiss
import numpy as np

In [24]:
resume_embeddings = np.array(
    resume_embeddings,
    dtype="float32"
)

In [25]:
embedding_dimension = resume_embeddings.shape[1]

print("Embedding dimension:", embedding_dimension)

Embedding dimension: 384


In [26]:
index = faiss.IndexFlatL2(embedding_dimension)

In [27]:
index.add(resume_embeddings)

In [28]:
print("Vectors stored:", index.ntotal)

Vectors stored: 2483


In [29]:
faiss.write_index(
    index,
    "../vector_db/resume_index.faiss"
)

In [30]:
import os

print(
    os.path.exists(
        "../vector_db/resume_index.faiss"
    )
)

True


## 6. Semantic Resume Search

In [31]:
def search_resumes(query, top_k=5):
    query_embedding = embedding_model.encode([query])

    query_embedding = np.array(
        query_embedding,
        dtype="float32"
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, resume_index in enumerate(indices[0]):
        candidate = rag_df.iloc[resume_index]

        results.append({
            "rank": rank + 1,
            "id": candidate["ID"],
            "category": candidate["Category"],
            "distance": distances[0][rank],
            "resume": candidate["clean_resume"]
        })

    return results

In [32]:
query = """
Find me a candidate with Python,
SQL, data analysis, and machine learning experience.
"""

results = search_resumes(query, top_k=5)

In [33]:
for result in results:
    print("=" * 80)
    print("Rank:", result["rank"])
    print("Candidate ID:", result["id"])
    print("Category:", result["category"])
    print("FAISS Distance:", result["distance"])
    print()
    print(result["resume"][:1000])
    print()

Rank: 1
Candidate ID: 62994611
Category: AGRICULTURE
FAISS Distance: 0.8822707

SOFTWARE DEVELOPER Professional Summary Enthusiastic computer engineer eager to contribute to team success through hard work, attention to detail and excellent organizational skills. Technical professional with complete understanding of entire software development life cycle. Respectful self-motivator gifted at finding reliable solutions for software issues. Experienced in c#, python, HTML, SQL, node.js/javascript and working knowledge of Restful API design & implementations. Fluent in English and Turkish and accustomed to working with cross-cultural, global teams. Skills C#, HTML, CSS, JavaScript, 5 years of experience SQL, 5 years of experience Python, MatLab, MongoDB, Tableau, Node JS Frameworks: .Net, Devexpress, TensorFlow, Keras, Scikit-learn, Pandas, NLTK. Search Engine Optimization Net API CSS Clients Database development Designing English HTML Image processing JavaScript Leadership Marketing MatLab

## 7. Display Retrieved Candidates

In [34]:
def display_results(results):
    for result in results:
        print("=" * 100)
        print(f"Rank: {result['rank']}")
        print(f"Candidate ID: {result['id']}")
        print(f"Category: {result['category']}")
        print(f"FAISS Distance: {result['distance']:.4f}")
        print("-" * 100)
        print(result["resume"][:1500])
        print()

In [35]:
display_results(results)

Rank: 1
Candidate ID: 62994611
Category: AGRICULTURE
FAISS Distance: 0.8823
----------------------------------------------------------------------------------------------------
SOFTWARE DEVELOPER Professional Summary Enthusiastic computer engineer eager to contribute to team success through hard work, attention to detail and excellent organizational skills. Technical professional with complete understanding of entire software development life cycle. Respectful self-motivator gifted at finding reliable solutions for software issues. Experienced in c#, python, HTML, SQL, node.js/javascript and working knowledge of Restful API design & implementations. Fluent in English and Turkish and accustomed to working with cross-cultural, global teams. Skills C#, HTML, CSS, JavaScript, 5 years of experience SQL, 5 years of experience Python, MatLab, MongoDB, Tableau, Node JS Frameworks: .Net, Devexpress, TensorFlow, Keras, Scikit-learn, Pandas, NLTK. Search Engine Optimization Net API CSS Clients Da

In [36]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

api_key = os.getenv("GEMINI_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


In [50]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [51]:
from src.llm_evaluator import llm

In [52]:
print(llm.model)

gemini-2.5-flash


In [53]:
import src.llm_evaluator

print(src.llm_evaluator.__file__)

e:\NLP\rag-talent-search-engine\src\llm_evaluator.py


In [54]:
import importlib
import src.llm_evaluator

importlib.reload(src.llm_evaluator)

llm = src.llm_evaluator.llm

print(llm.model)

gemini-3.6-flash


In [55]:
response = llm.invoke(
    "Reply with only: API connection successful"
)

print(response.content)

e:\NLP\rag-talent-search-engine\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'API connection successful', 'extras': {'signature': 'EpoECpcEARFNMg/GNRdqu4auBvfuh9Im+MTYWzPmGRBnQZg5naCEU4DU+bLZpcpPWyt7tTIy6xKmmtDAXHQOu2/zXh+1hUVOkZcCfWwJa+Se8btWa8yi68zmaxlRvkhI3xVPdoMtmEu7r2dc6+F0wmtwiSmm/Uw6gghucKu6xhKJvNGO/MofEM7t730W3i0wtE4jy7F8rtZZvlEwLqFJDHWwr/92riKE9S+plO71w54Q17PWcjzFDSzrV1nNku8aZxEiYhn3/bTOsHnLfYbuGHXbbPYPfXmE9RM8Gx8wQAa8QSBXSv7Vaq4ucr5x+pfRZ5zlxmuay9Xhchh2CjDhpNJup54Y8IVcq9V3fxW9RqHq/y4JjmIxx6ec4GULN/Qm+2zyYay7Vvsc3Wy/c8wR7Y5kHVcPs/POd/R2AzKPifzqjjH0ZGPu0fitlW9fygfI1N/7pXsx/DgCljEnAVj/2J82N62LTuGIG73OMBLk/4fYxCfvTwTcISMucXuu1efNtqfnvrGpuAXMyPq5p9hwhykaTWxTS59vd260gVtdKsY7MPBVI5WIgeOe4VvRz3YsB9z/Uy6rDJ2ulpG1SY5cDj89s8qCPgWH8UlYtu/sREZRgpoY0GphJZWfmC1rm1vnzo3jz24suXXD5pH1uSdeB5fl++vj3RvG28hdWTI/i4Erdc2mivRlJgxi794aio4TrtVAAjhcHHysPBkpCw=='}}]


In [56]:
response = llm.invoke(
    "Reply with only: API connection successful"
)

print(response.content[0]["text"])

e:\NLP\rag-talent-search-engine\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


API connection successful
